In [83]:
import sys
# Use an absolute path or relative path to the directory
sys.path.append("scripts")

import numpy as np
import pdb_voxelizier
import cnn_mlp_encoder
import jw_quantum_mapper
from scipy.optimize import minimize
from scipy.spatial.distance import squareform
from scipy.linalg import eigh
import sympy
import openfermion as op
import torch
import qiskit
import qiskit_algorithms
import qiskit_aer as q_aer

In [114]:
num_sites = 6
tensor = pdb_voxelizier.pdb_to_tensor('proteins/1ENH.pdb')
coefficients = cnn_mlp_encoder.get_hamiltonian(tensor, num_qubits=num_sites)
qubit_instructions = jw_quantum_mapper.apply_jw(coefficients,num_sites=num_sites)

# View instructions by uncommenting this
# jw_quantum_mapper.display_instructions(qubit_instructions)

In [ ]:
import pyvista as pv
def visualize_tensor(tensor):
    # [batch_size, channels, Depth, Height, Width]
    protein = tensor[0]
    
    protein = protein.to_dense().max(axis=0).values

    array = protein.numpy()
    grid = pv.wrap(array) # Automatically recognizes as a dataset
    grid.plot(jupyter_backend="client",volume=True)

torch_tensor = torch.from_numpy(tensor)
visualize_tensor(torch_tensor)

Widget(value='<iframe src="http://localhost:35407/index.html?ui=P_0x7f0f2c153390_1&reconnect=auto" class="pyvi…

Test our hamiltonian through a VQE!

In [126]:
# # To convert our list of strings into a PauliSum, we need to loop through each instruction
def convert_qubit_operators_to_pauli_operators(qubit_operators:list):
    coef_list = []
    op_list = []
    for i in qubit_operators:
        # Split up the operation into coefficient and the gates
        operation_list = i.split("*")
        
        # We grab the coefficient as a float and each operator as a single string   
        coef = float(operation_list[0])
        operators = operation_list[1].strip().split(" ")

        # We add all coefficients into a list
        coef_list.append(coef)

        
        # Go through each operator in the list, convert to Qiskit 'Pauli' term
        # First, we create a list of identity terms since each Pauli needs to be the same dimension
        start_op_list = list("I" * (num_sites))
        for term in operators:
            # First term is letter, next is integer
            start_op_list[int(term[1])] = term[0]
        op_list.append("".join(start_op_list))
    
    return qiskit.quantum_info.SparsePauliOp(data=op_list,coeffs=coef_list)

pauli_op = convert_qubit_operators_to_pauli_operators(qubit_instructions)

Get the ground state energy classically

In [116]:
# To define the matrix, we convert the pauli operator directly into a matrix
hamiltonian_matrix = pauli_op.to_matrix()

print("\nFinal Hamiltonian Matrix:")
print(hamiltonian_matrix)

# Now, we use scipy.linalg.eigh to calculate the ground state value of this matrix
lowest = min(eigh(hamiltonian_matrix)[0])

print(f"\nGround State Energy: {lowest}")



Final Hamiltonian Matrix:
[[ 0.0482+0.j  0.    +0.j  0.    +0.j ...  0.    +0.j  0.    +0.j
   0.    +0.j]
 [ 0.    +0.j -0.0196+0.j  0.023 +0.j ...  0.    +0.j  0.    +0.j
   0.    +0.j]
 [ 0.    +0.j  0.023 +0.j  0.0904+0.j ...  0.    +0.j  0.    +0.j
   0.    +0.j]
 ...
 [ 0.    +0.j  0.    +0.j  0.    +0.j ... -0.0904+0.j  0.023 +0.j
   0.    +0.j]
 [ 0.    +0.j  0.    +0.j  0.    +0.j ...  0.023 +0.j  0.0196+0.j
   0.    +0.j]
 [ 0.    +0.j  0.    +0.j  0.    +0.j ...  0.    +0.j  0.    +0.j
  -0.0482+0.j]]

Ground State Energy: -0.30720030160094375


Estimate ground state energy using VQE

In [117]:
# Now, lets create our ansatz circuit. In this case I use a variation of the hardware efficient Ansatz (HEA), specifically, qiskit's efficient_su2
from qiskit_algorithms.minimum_eigensolvers import AdaptVQE
from qiskit.circuit.library import EvolvedOperatorAnsatz

n = pauli_op.num_qubits
layers = 3
# ansatz = qiskit.circuit.library.efficient_su2(n, su2_gates=["ry"], entanglement="circular",reps=layers)
ansatz = EvolvedOperatorAnsatz(name="Ansatz_Adapt",reps=10)
# num_params = ansatz.num_parameters
# print(f"This ansatz has {num_params} parameters.")
# ansatz.decompose().draw("mpl",style="iqp")

Now we can run our circuit and attempt to converge on the ground state energy of the PauliSum.

In [118]:
from qiskit_algorithms.optimizers import SPSA
# Define Simulation
iterations = 1000
spsa = SPSA(maxiter=iterations)
counts = []
values = []

def store_intermediate_result(eval_count,parameters,mean,std):
    counts.append(eval_count)
    values.append(mean)

In [119]:
from qiskit_algorithms.utils import algorithm_globals
from qiskit_aer.primitives import EstimatorV2 as AerEstimator

seed = 170
algorithm_globals.random_seed = seed

noiseless_estimator = AerEstimator(options={"default_precision": 1e-2})


In [ ]:
from qiskit_algorithms import VQE
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms.optimizers import SLSQP


vqe = VQE(StatevectorEstimator(),ansatz=ansatz,optimizer=SLSQP())

result = vqe.compute_minimum_eigenvalue(operator=pauli_op)

print(result.eigenvalue)
print(lowest)

AlgorithmError: 'All gradients have been evaluated to lie below the convergence threshold during the first iteration of the algorithm. Try to either tighten the convergence threshold or pick a different ansatz.'